In [ ]:
import torch
import numpy as np
import warnings
import logging
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Suppress the expected model architecture warnings for clean logs
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

# ==========================================
# 1. SINGLETON MODEL INITIALIZATION
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if 'labse_model' not in globals():
    labse_model = SentenceTransformer('sentence-transformers/LaBSE').to(device)

if 'indicbert_model' not in globals():
    ib_id = "ai4bharat/IndicBERTv2-MLM-Sam-TLM"
    indicbert_tokenizer = AutoTokenizer.from_pretrained(ib_id)
    indicbert_model = AutoModel.from_pretrained(
        ib_id,
        output_hidden_states=True
    ).to(device)
    indicbert_model.eval()

# ==========================================
# 2. INDICBERT LAYER 8 POOLING
# ==========================================
def get_indicbert_word_embeddings(words, layer=8):
    """Extracts and averages subword embeddings from IndicBERT Layer 8."""

    tokens = indicbert_tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True
    ).to(device)

    word_ids = tokens.word_ids()

    with torch.no_grad():
        outputs = indicbert_model(**tokens)
        hidden_states = outputs.hidden_states[layer].squeeze(0).cpu().numpy()

    word_embeddings = []

    for i in range(len(words)):
        token_indices = [idx for idx, w_id in enumerate(word_ids) if w_id == i]

        if token_indices:
            word_emb = np.mean(hidden_states[token_indices], axis=0)
        else:
            word_emb = np.zeros(hidden_states.shape[1])

        word_embeddings.append(word_emb)

    return np.array(word_embeddings)

# ==========================================
# 3. THE CORE ENSEMBLE ENGINE
# ==========================================
def run_ensemble_mixer(eng_text, tam_text, semantic_weight=0.9, confidence_threshold=0.30):
    """
    Fuses LaBSE (semantic) and IndicBERT (structural) matrices
    and applies the recursive M:1 look-ahead heuristic.
    """

    e_words = eng_text.replace(".", "").replace(",", "").split()
    t_words = tam_text.replace(".", "").replace(",", "").split()

    # Extended functional word list for English-to-Tamil agglutination
    suffixes = {
        'in', 'the', 'a', 'an', 'on', 'at', 'to', 'of',
        'is', 'am', 'are', 'was', 'were', 'and',
        'towards', 'for', 'by', 'with'
    }

    # Step A: Semantic Extraction (LaBSE)
    e_emb_labse = labse_model.encode(e_words, convert_to_tensor=True).cpu().numpy()
    t_emb_labse = labse_model.encode(t_words, convert_to_tensor=True).cpu().numpy()
    sim_labse = cosine_similarity(e_emb_labse, t_emb_labse)

    # Step B: Structural Extraction (IndicBERT)
    e_emb_ib = get_indicbert_word_embeddings(e_words, layer=8)
    t_emb_ib = get_indicbert_word_embeddings(t_words, layer=8)
    sim_indicbert = cosine_similarity(e_emb_ib, t_emb_ib)

    # Step C: Matrix Fusion
    sim_mat = (semantic_weight * sim_labse) + ((1.0 - semantic_weight) * sim_indicbert)

    # Step D: Recursive M:1 Look-ahead Mapping
    links = []

    for i in range(len(e_words)):
        lookup_idx = i

        # Keep sliding forward as long as we hit functional/suffix words
        while lookup_idx < len(e_words) - 1 and e_words[lookup_idx].lower() in suffixes:
            lookup_idx += 1

        best_j = np.argmax(sim_mat[lookup_idx])
        score = sim_mat[lookup_idx][best_j]

        # Only append if we meet the minimum confidence
        if score > confidence_threshold:
            links.append({
                "english": e_words[i],
                "tamil": t_words[best_j],
                "confidence": round(float(score), 4)
            })

    return links

# ==========================================
# 4. MAIN PRODUCTION FUNCTION
# ==========================================
def align_sentence_pair(english_text, tamil_text):
    """
    Takes a pre-translated English-Tamil sentence pair and aligns the words.
    """

    if not english_text or not tamil_text:
        raise ValueError("Both English and Tamil text must be provided.")

    # Run the mixer
    links = run_ensemble_mixer(english_text, tamil_text, semantic_weight=0.9)

    # Format Output
    output = {
        "source_english": english_text,
        "target_tamil": tamil_text,
        "alignment_links": links
    }

    return output

# ==========================================
# EXECUTION BLOCK (For standalone testing)
# ==========================================
if __name__ == "__main__":
    # You provide both sentences manually here
    en_sentence = "I am on the ferry going to pick up the people."
    ta_sentence = "मैं लोगों को लेने के लिए नौका पर जा रहा हूँ।"

    print("⏳ Processing Alignment...")
    result = align_sentence_pair(en_sentence, ta_sentence)

    print(f"\n🇬🇧 English: {result['source_english']}")
    print(f"🇮🇳 Tamil:   {result['target_tamil']}\n")
    print(f"{'ENGLISH (M:1)':<20} | {'TAMIL TARGET':<25} | {'CONF'}")
    print("-" * 55)

    for link in result['alignment_links']:
        print(f"{link['english']:<20} | {link['tamil']:<25} | {link['confidence']}")

modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

...

The following layers were not sharded: embeddings.position_embeddings.weight, embeddings.word_embeddings.weight, encoder.layer.*.attention.output.LayerNorm.bias, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.attention.self.key.weight, pooler.dense.weight, encoder.layer.*.attention.self.query.bias, encoder.layer.*.attention.self.value.weight, encoder.layer.*.attention.output.dense.bias, encoder.layer.*.attention.self.query.weight, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.output.dense.bias, embeddings.LayerNorm.weight, embeddings.LayerNorm.bias, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.output.dense.weight, embeddings.token_type_embeddings.weight, encoder.layer.*.attention.self.value.bias, encoder.layer.*.attention.output.LayerNorm.weight, pooler.dense.bias, encoder.layer.*.attention.self.key.bias


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

⏳ Processing Alignment...


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]


🇬🇧 English: I am on the ferry going to pick up the people.
🇮🇳 Tamil:   मैं लोगों को लेने के लिए नौका पर जा रहा हूँ।

ENGLISH (M:1)        | TAMIL TARGET              | CONF
-------------------------------------------------------
I                    | मैं                       | 0.8842
am                   | नौका                      | 0.8118
on                   | नौका                      | 0.8118
the                  | नौका                      | 0.8118
ferry                | नौका                      | 0.8118
going                | जा                        | 0.8235
to                   | लेने                      | 0.7166
pick                 | लेने                      | 0.7166
up                   | पर                        | 0.6965
the                  | लोगों                     | 0.9574
people               | लोगों                     | 0.9574
